In [ ]:
# Operational setup only. No pip upgrades or model logic changes here.
import os
import sys
from pathlib import Path
_roots = [Path.cwd(), *Path.cwd().parents]
_work = next((p for p in _roots if (p / "revision_20261116/Source code/notebook_io.py").is_file()), None)
if _work is None:
    raise RuntimeError("Open from the course work directory or a child directory")
sys.path.insert(0, str(_work / "revision_20261116/Source code"))
os.environ["BASELINE_DISTANCE"] = "cosine"
import notebook_io as baseline_io
baseline_io.begin()


In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
# Replace Colab/Drive and live Google Sheets loading with the supplied local inputs.
import pandas as pd
import os
import zipfile
DOCUMENTS_PATH = baseline_io.prepare_inputs()
df = pd.read_csv(baseline_io.RES_CSV)
print(f"Fixed evaluation pairs: {len(df)}")


In [ ]:
from langchain_community.vectorstores.utils import DistanceStrategy
if os.environ.get("BASELINE_REUSE_INDEX_FROM"):
    baseline_io.reuse_index()
else:
    from langchain_community.vectorstores.utils import DistanceStrategy
    import os
    from tqdm import tqdm
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.document_loaders import TextLoader
    from langchain_community.vectorstores import FAISS
    from langchain_community.embeddings import HuggingFaceEmbeddings

    # === Configuration ===
    DOCUMENTS_PATH = str(baseline_io.CORPUS_ROOT)
    EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
    BATCH_SIZE = 1024
    OUTPUT_DIR = str(baseline_io.INDEX_DIR)

    # === Load text documents ===
    def load_text_documents(path):
      documents = []
      for filename in baseline_io.document_filenames(path):
          if filename.endswith(".txt"):
              file_path = os.path.join(path, filename)
              loader = TextLoader(file_path, encoding='utf-8')
              raw_docs = loader.load()
              for doc in raw_docs:
                  # Normalize content
                  content = doc.page_content.replace("\n", " ").lower().strip()
                  doc.page_content = content
                  documents.append(doc)
      return documents

    print("Loading text documents...")
    raw_documents = load_text_documents(DOCUMENTS_PATH)
    print(f"Total documents found: {len(raw_documents)}")

    subset_idx = len(raw_documents)
    raw_documents = raw_documents[:subset_idx]
    print(f"Documents selected for embedding: {len(raw_documents)}")

    # === Split documents into smaller chunks ===
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=50,
        length_function=len
    )

    texts = []
    for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
        chunks = text_splitter.split_documents([doc])
        texts.extend(chunks)

    print(f"Total chunks created: {len(texts)}")

    # === Initialize embeddings ===
    print(f"Loading embedding model: {EMBEDDING_MODEL}")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cuda", "revision": baseline_io.EMBEDDING_REVISION}
    )

    # === Build FAISS vectorstore ===
    vectorstore = None

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to FAISS"):
        batch = texts[i:i + BATCH_SIZE]
        batch_embeddings = embeddings.embed_documents([t.page_content for t in batch])

        if vectorstore is None:
            vectorstore = FAISS.from_texts(
                [t.page_content for t in batch],
                embeddings,
                normalize_L2=True,
                distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
            )
        else:
            vectorstore.add_texts(
                [t.page_content for t in batch],
                embedding=batch_embeddings
            )

    # === Save FAISS index ===
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    vectorstore.save_local(OUTPUT_DIR)
    print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'")
    baseline_io.record_index(vectorstore, len(raw_documents), len(texts))


In [ ]:
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# === Configuration ===
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
VECTORSTORE_PATH = str(baseline_io.INDEX_DIR)

# === Load FAISS vectorstore ===
print("Loading FAISS vectorstore...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu", "revision": baseline_io.EMBEDDING_REVISION}
)

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True,
    normalize_L2=True,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
)
print("FAISS vectorstore loaded successfully.")

# === Display a sample record ===
keys = list(vectorstore.docstore._dict.keys())
if not keys:
    raise ValueError("The FAISS docstore is empty. No records found.")

first_key = keys[0]
record = vectorstore.docstore._dict[first_key]

print("\n--- First Record ---")
print(f"ID: {first_key}")
print(f"Content (truncated): {record.page_content[:200]}...")
print(f"Metadata: {record.metadata}")

# === Retrieve the corresponding vector ===
index_id = 0  # first vector
vector = vectorstore.index.reconstruct(index_id)

print("\n--- Corresponding Vector ---")
print(f"Vector size: {len(vector)}")
print(f"First 5 values: {vector[:5]}")


In [ ]:
import os
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
# Unused Google API import omitted; this notebook does not call an external LLM API.
from sklearn.metrics.pairwise import cosine_similarity

# ==================== LOAD DATA ====================
df = pd.read_csv(baseline_io.RES_CSV)
df = df.head(baseline_io.N)
assert len(df) == baseline_io.N and (len(df) >= 100 or baseline_io.IS_PILOT)
print(f"Total number of questions: {len(df)}")

# # ==================== LOAD FAISS + EMBEDDINGS ====================
# embedding_model_name = "keepitreal/vietnamese-sbert"
# embeddings = HuggingFaceEmbeddings(
#     model_name=embedding_model_name,
#     model_kwargs={"device": "cpu"}  # or "cuda"
# )

vectorstore = FAISS.load_local(
    str(baseline_io.INDEX_DIR),
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
    normalize_L2=True,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
)

# ==================== LOAD LLM ====================
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen1.5-1.8B", #Choose your model
    task="text-generation",
    device=0,  # GPU 0
    model_kwargs={"temperature": 0, "max_length": 1024, "revision": baseline_io.LLM_REVISION},
    pipeline_kwargs={"framework": "pt"}
)

baseline_io.record_model(llm, embeddings)

# ==================== HELPER FUNCTIONS ====================
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question, k=k)
    context_texts = []
    for doc in docs:
        if hasattr(doc, "page_content"):
            context_texts.append(doc.page_content)
        elif isinstance(doc, dict) and "page_content" in doc:
            context_texts.append(doc["page_content"])
        else:
            context_texts.append(str(doc))
    context = "\n".join(context_texts)

    prompt = f"""
Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.
Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.

Ngữ cảnh:
{context}

Câu hỏi: {question}
Trả lời ngắn gọn và chính xác nhất có thể:
"""

    baseline_io.capture_context(question, docs, prompt)
    output = llm.generate([prompt],  max_new_tokens=256)
    generated_text = output.generations[0][0].text
    print(generated_text)
    return generated_text

def cosine_similarity_score(pred: str, true: str, emb_model) -> float:
    if not pred or not true:
        return 0.0
    pred_vec = emb_model.embed_query(pred)
    true_vec = emb_model.embed_query(true)
    return float(cosine_similarity([pred_vec], [true_vec])[0][0])

def jaccard_similarity(pred: str, true: str) -> float:
    pred_words = set(normalize_text(pred).split())
    true_words = set(normalize_text(true).split())
    return len(pred_words & true_words) / len(pred_words | true_words) if pred_words and true_words else 0.0

def token_overlap_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not true_words:
        return 0.0
    return sum((Counter(pred_words) & Counter(true_words)).values()) / len(true_words)

def bleu_score_simple(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    overlap = sum((Counter(pred_words) & Counter(true_words)).values())
    precision = overlap / len(pred_words)
    bp = 1.0 if len(pred_words) >= len(true_words) else np.exp(1 - len(true_words)/len(pred_words))
    return bp * precision

def rouge_l_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    m, n = len(pred_words), len(true_words)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if pred_words[i-1] == true_words[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / m
    recall = lcs / n
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def semantic_containment(pred: str, true: str) -> float:
    pred_norm = normalize_text(pred)
    true_norm = normalize_text(true)
    if not true_norm:
        return 0.0
    if true_norm in pred_norm:
        return 1.0
    pred_words = set(pred_norm.split())
    true_words = set(true_norm.split())
    return 1.0 if true_words.issubset(pred_words) else 0.0

# ==================== EVALUATION ====================
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RAG"):
    question = row["Câu Hỏi"]
    ground_truth = str(row["Trả lời"]).strip()

    try:
        response = rag_query(question)
    except Exception as e:
        baseline_io.capture_error(question)
        print(f"Error at question {idx}: {e}")
        response = ""

    metrics = {
        "Cosine_Similarity": cosine_similarity_score(response, ground_truth, embeddings),
        "Jaccard_Similarity": jaccard_similarity(response, ground_truth),
        "Token_Overlap": token_overlap_score(response, ground_truth),
        "BLEU_Score": bleu_score_simple(response, ground_truth),
        "ROUGE_L": rouge_l_score(response, ground_truth)
    }

    baseline_io.record_result(idx, row, response, metrics)
    results.append(metrics)

results_df = pd.DataFrame(results)
mean_metrics = results_df.mean().to_dict()

print("\n=== FINAL RAG EVALUATION SUMMARY ===")
for k, v in mean_metrics.items():
    print(f"{k}: {v:.4f}")

results_df.to_csv(baseline_io.RUN / "rag_baseline_metrics.csv", index=False)
print("Metrics saved to rag_baseline_metrics.csv")

baseline_io.finish(results_df, mean_metrics, normalize_text)
